## Council tax by London borough

The listings dataset records council tax as "Ask agent" in 99.1% of cases, so
the field was dropped during cleaning. Following supervisory guidance, council
tax is instead taken from the MHCLG statistical release on council tax levels
set by local authorities in England, joined to listings through the Local
Authority District code already present in the spatial lookup.

Band D is used as the reference band, as published. The listings do not record
a property's band, so a band-specific figure cannot be derived without an
additional assumption. Band D is the statutory reference point and is the band
against which all others are proportioned.

Figures include adult social care and parish precepts, reflecting what a
household actually pays. Monthly equivalents divide the annual charge by twelve.

Source: MHCLG, Council Tax levels set by local authorities in England 2026 to
2027, Table 8a. Published 25 March 2026. Open Government Licence v3.0.
Output: `data/processed/council_tax.csv`

In [2]:
import pandas as pd

ct = pd.read_excel('../data/raw/Tables_1-9_2026-27.ods', 'Table_8a',
                   engine='odf', header=None)
print(ct.head(15).to_string())

                                                                                                                                                                                                                                                 0          1                       2      3       4             5                                                                                                       6                                                                                                                              7                                                                                                8                                                                                                                      9                                                         10                                                                             11     12
0                                                                                           Table 8a:

In [3]:
# Real headers are on row 2 (zero-indexed); rows 0-1 are title and notes
ct = pd.read_excel('../data/raw/Tables_1-9_2026-27.ods', 'Table_8a',
                   engine='odf', header=2)

# Keep only London boroughs and the columns needed
london = ct[ct['Region'] == 'L'].copy()

london = london[['ONS Code', 'Authority', 'Area',
                 'Average council tax for the authority including adult social care and parish precepts (Band D)£']]
london.columns = ['lad_code', 'authority', 'area', 'band_d_annual']

london['band_d_monthly'] = (london['band_d_annual'] / 12).round(2)

print(london.shape)
print(london.head(10).to_string(index=False))

(36, 5)
 lad_code              authority         area  band_d_annual  band_d_monthly
E09000007                 Camden Inner London        1697.04          141.42
E09000001         City of London Inner London        1153.18           96.10
E09000011              Greenwich Inner London        1597.35          133.11
E09000012                Hackney Inner London        1549.79          129.15
E09000013 Hammersmith and Fulham Inner London        1009.00           84.08
E09000019              Islington Inner London        1597.64          133.14
E09000020 Kensington and Chelsea Inner London        1156.14           96.35
E09000022                Lambeth Inner London        1536.60          128.05
E09000023               Lewisham Inner London        1726.82          143.90
E09000028              Southwark Inner London        1456.75          121.40


In [4]:
print(london['area'].value_counts())
print()
print("Lowest: ", london.nsmallest(3, 'band_d_monthly')[['authority', 'band_d_monthly']].to_string(index=False))
print("Highest:", london.nlargest(3, 'band_d_monthly')[['authority', 'band_d_monthly']].to_string(index=False))

area
Outer London                20
Inner London                13
Greater London Authority     3
Name: count, dtype: int64

Lowest:                               authority  band_d_monthly
               Of which other services           14.70
Of which metropolitan police authority           27.84
              Greater London Authority           42.54
Highest:            authority  band_d_monthly
Kingston upon Thames          174.89
             Croydon          174.12
              Harrow          166.71


Three Greater London Authority rows are excluded, as these are precept
components already included in each borough's total rather than separate
billing authorities.

In [5]:
# The three Greater London Authority rows are precept components already
# included in each borough's total, not separate billing authorities
london = london[london['area'] != 'Greater London Authority']

print("Boroughs:", len(london))
print()
print("Lowest: ", london.nsmallest(3, 'band_d_monthly')[['authority', 'band_d_monthly']].to_string(index=False))
print("Highest:", london.nlargest(3, 'band_d_monthly')[['authority', 'band_d_monthly']].to_string(index=False))

london.to_csv('../data/processed/council_tax.csv', index=False)
print("\nSaved:", london.shape)

Boroughs: 33

Lowest:               authority  band_d_monthly
            Wandsworth           43.14
           Westminster           44.92
Hammersmith and Fulham           84.08
Highest:            authority  band_d_monthly
Kingston upon Thames          174.89
             Croydon          174.12
              Harrow          166.71

Saved: (33, 5)
